# Claude Code in Action

Notes from the Anthropic Academy course *Claude Code in Action*: running long,
hands-off Claude Code sessions you can trust.
Sections marked ★ are the highest-value material.

## Table of Contents

1. Lecture 1 - Steering Long Sessions
2. Lecture 2 - A CLAUDE.md That Follows
3. Lecture 3 - Verification Skills
4. Lecture 4 - Permission Modes
5. Lecture 5 - Hooks
6. Lecture 6 - Routines and Headless
7. Lecture 7 - GitHub Actions and Code Review
8. Lecture 8 - Trust It: Verifying Unsupervised Runs
9. Lecture 9 - Plugins

**The through-line of the whole course:** instructions are things Claude usually
follows; hooks are code that always runs. Anything you cannot afford to have
skipped belongs in a hook, not in a CLAUDE.md file and not in a skill.

## ★ Executive Summary

Long sessions come down to two habits: **scope the work before Claude starts, and
steer it while it runs.**

- **Scope** with plan mode. Claude researches read-only and hands you a plan. Read
  it properly. Iterating on a plan is far cheaper than cleaning up a bad run.
- **Steer with `/compact <instructions>`.** Never run bare `/compact`. Whatever you
  write after the command shapes what the summary keeps.
- **Course-correct with rewind.** Double tap escape on an empty prompt. Every user
  prompt is a checkpoint. Five options: restore code, restore conversation, restore
  both, summarize from here, summarize up to here.
- **Hand off with `/goal`** when you can describe "done" better than the steps. A
  fast evaluator checks the condition each turn. It only reads the transcript, so
  the condition must be checkable from output Claude produces.
- **`/loop`** runs a prompt on an interval between turns, for watching external
  state like a CI run. Escape stops it.
- **Run parallel agents in worktrees.** Each session gets its own file tree, so
  two agents cannot clobber each other. Clean worktrees are removed on exit.

# Lecture 1 - Steering Long Sessions

### ★ Scope the work first with plan mode

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1380 605" width="880" height="386" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img"><title>Scoping and steering a long session</title><desc>Three step flow from plan mode to execution, with three steering tools branching below.</desc><rect width="1380" height="605" fill="#FFFFFF"/><defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs><text x="95" y="99" font-size="46" font-weight="700" fill="#000000">1</text><rect x="60" y="115" width="300" height="170" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/><text x="210" y="165" font-size="19" fill="#000000" text-anchor="middle"><tspan x="210" dy="0" font-weight="700">Plan mode</tspan><tspan x="210" dy="28">Claude researches in</tspan><tspan x="210" dy="28">read-only and hands</tspan><tspan x="210" dy="28">you a plan</tspan></text><line x1="360" y1="200" x2="522" y2="200" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/><text x="565" y="99" font-size="46" font-weight="700" fill="#000000">2</text><rect x="530" y="115" width="300" height="170" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/><text x="680" y="179" font-size="19" fill="#000000" text-anchor="middle"><tspan x="680" dy="0" font-weight="700">Actually read it</tspan><tspan x="680" dy="28">Iterating on a plan beats</tspan><tspan x="680" dy="28">cleaning up a bad run</tspan></text><line x1="830" y1="200" x2="992" y2="200" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><text x="1035" y="99" font-size="46" font-weight="700" fill="#000000">3</text><rect x="1000" y="115" width="300" height="170" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/><text x="1150" y="193" font-size="19" fill="#000000" text-anchor="middle"><tspan x="1150" dy="0" font-weight="700">Claude executes</tspan><tspan x="1150" dy="28">the approved plan</tspan></text><path d="M 1150 285 L 1150 330" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round"/><line x1="250" y1="330" x2="1150" y2="330" stroke="#000000" stroke-width="3"/><line x1="250" y1="330" x2="250" y2="387" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><line x1="690" y1="330" x2="690" y2="387" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><line x1="1130" y1="330" x2="1130" y2="387" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="604.84" y="303" width="190.32" height="24" fill="#FFFFFF"/><text x="700" y="322" font-size="16" fill="#5F6368" text-anchor="middle">steer while it runs</text><rect x="60" y="395" width="380" height="150" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/><text x="250" y="449" font-size="19" fill="#000000" text-anchor="middle"><tspan x="250" dy="0" font-weight="700">/compact &lt;focus&gt;</tspan><tspan x="250" dy="28">Anything after the command</tspan><tspan x="250" dy="28">shapes what the summary keeps</tspan></text><rect x="500" y="395" width="380" height="150" rx="8" fill="#DDF7F1" stroke="#12807A" stroke-width="3"/><text x="690" y="449" font-size="19" fill="#000000" text-anchor="middle"><tspan x="690" dy="0" font-weight="700">Rewind</tspan><tspan x="690" dy="28">Double tap escape to reach</tspan><tspan x="690" dy="28">your last checkpoint</tspan></text><rect x="940" y="395" width="380" height="150" rx="8" fill="#EEEEFC" stroke="#5A4FF0" stroke-width="3"/><text x="1130" y="449" font-size="19" fill="#000000" text-anchor="middle"><tspan x="1130" dy="0" font-weight="700">/goal and /loop</tspan><tspan x="1130" dy="28">Hand off when you can describe</tspan><tspan x="1130" dy="28">done better than the steps</tspan></text></svg>

*Scope once with plan mode, then reach for a steering tool while the run is in flight.*

1. Start in **plan mode**. Claude works read-only: it reads the code and works out
   what needs to change without editing anything.
2. Claude hands you a plan.
3. **Actually read it. Do not skim it.** The more thorough the plan, the fewer
   surprises during execution.
4. If something is off or missing, ask Claude to add it where you want it.
5. Only then let Claude execute.

The economics are the point: iterating on a plan is much faster than letting Claude
run, hoping for the best, and cleaning up the mess.

### ★ Compact: direct the summary, never run it bare

Compaction summarizes the conversation, makes that summary the new context, and
deletes the old messages. The risk is that something important gets dropped and
Claude drifts.

The fix is to put instructions after the command. Anything you write there shapes
what the summary keeps. That is your steering wheel for context.

In [ ]:
# Bad: no direction, Claude decides what matters
/compact

# Good: name what the summary must preserve
/compact Focus on the --version flag implementation

### Rewind: the checkpoint menu

When Claude heads down the wrong path, do not prompt your way back out. **Double tap
escape on an empty prompt** to open the rewind menu. Every user prompt creates a
checkpoint.

| Option | What it does | When to use it |
|---|---|---|
| Restore code and conversation | Rolls back both together | Full undo of a bad stretch |
| Restore conversation | Rolls back the chat only | Keep the files, drop the reasoning |
| Restore code | Rolls back the files only | Keep the discussion, drop the edits |
| Summarize from here | Summarizes everything **after** the checkpoint | You had a side conversation and want the space back |
| Summarize up to here | Summarizes everything **before** the checkpoint | Long setup phase to compress, implementation to keep intact |

### Let Claude run more autonomously: goal and loop

Everything above assumes you are hands-on. `/goal` and `/loop` are the autonomous
alternatives.

**`/goal`** sets a completion condition. You describe what "done" looks like and
Claude keeps working across turns until a fast evaluator confirms the conditions are
met. It will not stop the first time it thinks it has finished.

**★ The constraint that matters:** the evaluator only reads the transcript. Your
condition has to be checkable from output Claude actually produces, such as the
result of a test run. "The code is well designed" is not a usable goal condition.

**`/loop`** runs a prompt on an interval between turns, either fixed or self-paced.
Use it to pull external state (a CI run, a deploy) and act when it changes. Press
escape to stop it.

In [ ]:
# Set a completion condition the evaluator can actually check from the transcript
/goal all tests in src/billing pass, and the type checker reports zero errors

# Cancel it
/goal clear

### Run parallel work in worktrees

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1240 560" width="880" height="397" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img"><title>Worktrees isolate parallel sessions</title><desc>One shared repository fans out into three isolated worktrees, one per Claude session.</desc><rect width="1240" height="560" fill="#FFFFFF"/><defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs><rect x="470" y="60" width="300" height="110" rx="8" fill="#EBD6F8" stroke="#A81FEE" stroke-width="3"/><text x="620" y="122" font-size="19" fill="#000000" text-anchor="middle"><tspan x="620" dy="0" font-weight="700">Shared git repository</tspan></text><path d="M 620 170 L 620 215" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round"/><line x1="250" y1="215" x2="990" y2="215" stroke="#000000" stroke-width="3"/><line x1="250" y1="215" x2="250" y2="292" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><line x1="620" y1="215" x2="620" y2="292" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><line x1="990" y1="215" x2="990" y2="292" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="60" y="250" width="1120" height="250" rx="8" fill="none" stroke="#6BB8F5" stroke-width="2" stroke-dasharray="8 6"/><rect x="72" y="260" width="356.1" height="21" fill="#FFFFFF"/><text x="78" y="276" font-size="16" fill="#5F6368">One independent file tree per session</text><rect x="100" y="300" width="300" height="150" rx="8" fill="#DDF7F1" stroke="#12807A" stroke-width="3"/><text x="250" y="354" font-size="19" fill="#000000" text-anchor="middle"><tspan x="250" dy="0" font-weight="700">Session A</tspan><tspan x="250" dy="28">own worktree,</tspan><tspan x="250" dy="28">own file tree</tspan></text><rect x="470" y="300" width="300" height="150" rx="8" fill="#DDF7F1" stroke="#12807A" stroke-width="3"/><text x="620" y="354" font-size="19" fill="#000000" text-anchor="middle"><tspan x="620" dy="0" font-weight="700">Session B</tspan><tspan x="620" dy="28">own worktree,</tspan><tspan x="620" dy="28">own file tree</tspan></text><rect x="840" y="300" width="300" height="150" rx="8" fill="#DDF7F1" stroke="#12807A" stroke-width="3"/><text x="990" y="354" font-size="19" fill="#000000" text-anchor="middle"><tspan x="990" dy="0" font-weight="700">Session C</tspan><tspan x="990" dy="28">own worktree,</tspan><tspan x="990" dy="28">own file tree</tspan></text></svg>

*Each parallel session gets its own file tree, so two agents cannot clobber each other's changes.*

Two Claude sessions fighting over the same files leads to conflicts. Worktrees give
each session its own independent file tree.

- Because each agent has its own tree, they cannot overwrite each other.
- When a session exits, a **clean** worktree is removed automatically.
- A **`.worktreeinclude`** file at the repo root lists git-ignored files to copy into
  every worktree. This is how you get an environment variable file or a local config
  into each tree without committing it to version control.

## Executive Summary

**CLAUDE.md is guidance, not enforced configuration.** Every line competes with every
other line for attention. The longer the file, the less reliably Claude follows any
single rule. Lean file, more of it followed.

- **Before writing a rule, ask if it belongs here at all.** Hard lines ("never push
  to main") belong in a PreToolUse hook, which is code that can actually block the
  action. CLAUDE.md handles softer conventions.
- **Four locations, all loaded at launch:** managed policy (org, cannot be excluded),
  user (your machine), project (shared, in the repo), local (git-ignored, just you).
- **Imports organise, they do not shrink context.** `@path/to/file.md` is expanded
  inline at launch. Everything still loads.
- **Phrasing determines compliance.** Be specific and checkable. Name the replacement
  instead of only banning something.
- **Emphasis is a budget.** "IMPORTANT" and "YOU MUST" raise priority only relative to
  quieter text around them. If everything shouts, nothing does. Spend it on two or
  three rules.
- **Treat a violation as a bug report against the file.** Say "add that to the
  CLAUDE.md file" and Claude writes the rule.

# Lecture 2 - A CLAUDE.md That Follows

### ★ Guidance vs enforcement: is CLAUDE.md even the right tool?

A growing CLAUDE.md is the trap that catches almost everyone: you hit a problem, you
add a rule, and eventually Claude starts ignoring parts of the file. That is not a
bug in Claude, it is how the file works.

Take "never push to main". Put it in CLAUDE.md and you are **hoping** Claude reads and
respects it. Most of the time it will, and "most of the time" is not good enough for
something that dangerous. A **pre-tool-use hook** is code that runs before Claude acts
and can block the action outright. That is real enforcement, not a polite request.

### The four locations

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1240 700" width="880" height="497" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img"><title>The four CLAUDE.md locations</title><desc>Four CLAUDE.md files at managed policy, user, project and local scope all load together at launch.</desc><rect width="1240" height="700" fill="#FFFFFF"/><defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs><rect x="60" y="60" width="460" height="130" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/><text x="290" y="104" font-size="19" fill="#000000" text-anchor="middle"><tspan x="290" dy="0" font-weight="700">Managed policy</tspan><tspan x="290" dy="28">Org level, set by your</tspan><tspan x="290" dy="28">platform team. Cannot be excluded.</tspan></text><path d="M 520 125 L 620 125 L 620 350" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round"/><rect x="60" y="210" width="460" height="130" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/><text x="290" y="254" font-size="19" fill="#000000" text-anchor="middle"><tspan x="290" dy="0" font-weight="700">User</tspan><tspan x="290" dy="28">Your preferences, on every</tspan><tspan x="290" dy="28">project on your machine.</tspan></text><path d="M 520 275 L 620 275 L 620 350" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round"/><rect x="60" y="360" width="460" height="130" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/><text x="290" y="404" font-size="19" fill="#000000" text-anchor="middle"><tspan x="290" dy="0" font-weight="700">Project</tspan><tspan x="290" dy="28">Shared with the team,</tspan><tspan x="290" dy="28">checked into the repo.</tspan></text><path d="M 520 425 L 620 425 L 620 350" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round"/><rect x="60" y="510" width="460" height="130" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/><text x="290" y="554" font-size="19" fill="#000000" text-anchor="middle"><tspan x="290" dy="0" font-weight="700">Local</tspan><tspan x="290" dy="28">Git ignored. Just you,</tspan><tspan x="290" dy="28">just this one repo.</tspan></text><path d="M 520 575 L 620 575 L 620 350" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round"/><line x1="620" y1="350" x2="792" y2="350" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/><rect x="604.84" y="323" width="190.32" height="24" fill="#FFFFFF"/><text x="700" y="342" font-size="16" fill="#5F6368" text-anchor="middle">all four, at launch</text><rect x="800" y="275" width="380" height="150" rx="8" fill="#BDF0E4" stroke="#12807A" stroke-width="3"/><text x="990" y="329" font-size="19" fill="#000000" text-anchor="middle"><tspan x="990" dy="0" font-weight="700">Loaded together</tspan><tspan x="990" dy="28">Nothing is dropped.</tspan><tspan x="990" dy="28">They stack.</tspan></text></svg>

*All four files load together at launch and stack; none of them is dropped.*

| Location | Scope | Notes |
|---|---|---|
| **Managed policy** | Org level, set by your platform team | You cannot exclude it, so org policy is always in play |
| **User** | You, across every project on the machine | Personal preferences that follow you |
| **Project** | Your team | Checked into the repo |
| **Local** | You, this repo only | Ignored by git |

**Local is the one people overlook.** Refactoring on your own branch and want Claude
to hold some architectural decisions in mind? That does not belong in the shared
project file where it would affect the whole team. Put it in local.

### Imports organise, they do not shrink context

When the project file gets long, split it with the path-to-file import syntax.

In [ ]:
@.claude/conventions/code-style.md
@.claude/conventions/testing.md
@.claude/conventions/workflow.md

**Know exactly what this buys you.** At launch, Claude expands the imported files
inline, right where you referenced them. Imports keep things tidy. They do **not**
reduce the amount of context Claude has to read. Use them to organise, not to shrink
the load.

### ★ Phrasing is what makes rules stick

Most rules fail because they are vague. Two fixes:

**1. Be specific and checkable.** If you cannot check whether a rule was followed,
neither can Claude.

| Vague | Specific |
|---|---|
| "Follow best practices for API routes." | "Put new API routes in `src/api/handlers`, one per file." |

**2. Name the replacement, do not just ban something.** Banning leaves the door open.

| Leaves it open | Closes it |
|---|---|
| "Don't use default exports." | "Use named exports, not default exports." |

**3. Emphasis is a budget.** `IMPORTANT` and `YOU MUST` raise a rule's priority only
relative to everything quieter around it. If every rule shouts, the emphasis means
nothing. Spend it on the two or three rules that really hurt when broken.

### Keep the file under revision

The file is never finished. Treat it like production code: **if you cannot justify a
line, delete it.**

When Claude does the wrong thing, do not sigh and fix it by hand. Treat it as a bug
report against your CLAUDE.md. You can tell Claude directly, "add that to the
CLAUDE.md file", and it will write the rule for you. The file improves every time
something goes wrong.

## Executive Summary

**If you build one skill first, build a verification skill.** Normally, checking
Claude's work depends on you remembering to ask. A verification skill removes that
dependency: it fires on its own when the change matches its description.

The chain it runs: run the test suite, read the diff, **confirm no test was weakened
just to make things pass**, then report pass or fail with the evidence attached.

- Green tests are not proof. A test can be quietly loosened so it passes regardless.
  That is why reading the diff is part of the chain.
- "Done" is not "the code looks right". Done is **the gates being run and observed,
  with the results stated explicitly.**
- **Rule of thumb: if you have typed the same multi-step instruction twice, that is a
  skill.** Release checklists, migration recipes, pre-PR checks.
- **A skill folder holds more than instructions.** A `reference.md` for depth Claude
  reads only when needed, and scripts Claude executes rather than loads into context.
  Keep `skill.md` itself lean.
- Only skill **descriptions** load into context until a skill is needed, so there is
  no cost to packaging every procedure you repeat.

# Lecture 3 - Verification Skills

### ★ Why verification is the skill to build first

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1300 670" width="880" height="454" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img"><title>How a verification skill fires</title><desc>Five step flow: a refactor triggers the skill, which runs tests, reads the diff and reports with evidence.</desc><rect width="1300" height="670" fill="#FFFFFF"/><defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs><text x="95" y="124" font-size="46" font-weight="700" fill="#000000">1</text><rect x="60" y="140" width="340" height="150" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/><text x="230" y="208" font-size="19" fill="#000000" text-anchor="middle"><tspan x="230" dy="0" font-weight="700">You ask Claude</tspan><tspan x="230" dy="28">to refactor something</tspan></text><text x="515" y="124" font-size="46" font-weight="700" fill="#000000">2</text><rect x="480" y="140" width="340" height="150" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/><text x="650" y="194" font-size="19" fill="#000000" text-anchor="middle"><tspan x="650" dy="0">The change matches the</tspan><tspan x="650" dy="28">skill's description, so</tspan><tspan x="650" dy="28" font-weight="700">the skill fires on its own</tspan></text><text x="935" y="124" font-size="46" font-weight="700" fill="#000000">3</text><rect x="900" y="140" width="340" height="150" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/><text x="1070" y="208" font-size="19" fill="#000000" text-anchor="middle"><tspan x="1070" dy="0">Runs the</tspan><tspan x="1070" dy="28" font-weight="700">full test suite</tspan></text><line x1="400" y1="215" x2="472" y2="215" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><line x1="820" y1="215" x2="892" y2="215" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><path d="M 1070 290 L 1070 375 L 230 375 L 230 452" fill="none" stroke="#000000" stroke-width="7" stroke-linejoin="round" marker-end="url(#ah)"/><text x="95" y="444" font-size="46" font-weight="700" fill="#000000">4</text><rect x="60" y="460" width="340" height="150" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/><text x="230" y="514" font-size="19" fill="#000000" text-anchor="middle"><tspan x="230" dy="0">Reads the diff and checks</tspan><tspan x="230" dy="28" font-weight="700">no test was weakened</tspan><tspan x="230" dy="28">just to make things pass</tspan></text><text x="515" y="444" font-size="46" font-weight="700" fill="#000000">5</text><rect x="480" y="460" width="340" height="150" rx="8" fill="#DDF7F1" stroke="#12807A" stroke-width="3"/><text x="650" y="528" font-size="19" fill="#000000" text-anchor="middle"><tspan x="650" dy="0" font-weight="700">Reports pass or fail</tspan><tspan x="650" dy="28">with the evidence attached</tspan></text><line x1="400" y1="535" x2="472" y2="535" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/></svg>

*The skill's description is the trigger; once fired it walks the same four steps every time.*

The problem with checking Claude's work manually is that the checking depends on you
remembering to ask for it. Skip that step once and bad code slips through.

The flow, with no prompting from you:

1. You ask Claude to refactor something.
2. The change matches the skill's description, so the **skill fires on its own**.
3. It runs the test suite.
4. It reads the diff and confirms **no test was weakened just to make things pass**.
5. It reports pass or fail, with the evidence attached.

**★ Note step 4.** It is not enough to run the tests and see green, because a test can
be quietly loosened so it passes no matter what. Done is the gates being run and
observed, with results stated explicitly, not "the diff looks right".

This shape carries any procedure your team repeats: a release checklist, a migration
recipe, a pre-PR check.

### A skill folder can hold more than instructions

A skill is not just a single `skill.md`. The folder around it carries more, and that
is what makes skills powerful for verification.

| What you put in the folder | How Claude treats it |
|---|---|
| `skill.md` | The lean procedure. Name, description (the trigger), steps. |
| `reference.md` | Detailed material linked from `skill.md`. **Read only when the depth is actually needed.** |
| Scripts (e.g. `check.sh`) | **Executed, not loaded into context.** A skill can carry its own tooling. |

The takeaway: keep `skill.md` lean. Push heavy material, long explanations and
executable scripts into side files. The lean file describes what to do; the side
files hold the depth and the tools.

### ★ Which instruction surface owns which rule

Three places to put instructions, easy to mix up:

| Surface | Owns | Nature |
|---|---|---|
| **CLAUDE.md** | Conventions that apply all the time: naming rules, where files go | Instruction Claude follows |
| **Skill** | Procedures and reference material tied to a particular kind of task | Instruction Claude follows |
| **Hook** | Anything Claude **must not be able to skip** | Code that actually runs |

**If skipping the rule is not acceptable, do not leave it to instruction-following.**

Build the verification skill, check it into your project's `.claude/skills`, and the
whole team inherits the same move. Everyone's work gets checked the same way, without
anyone having to remember to ask.

## Executive Summary

Permission modes let you decide **once** what Claude may run without asking, instead
of approving action by action. **Shift-tab** cycles the everyday modes and the status
bar shows the current one.

- **Six modes:** manual, accept edits, plan, auto, don't ask, bypass permissions.
- **Auto is the hands-off mode.** A separate classifier model reviews each action
  before it runs. It **guards intent**: it blocks production deploys and migrations,
  force pushes, piping downloaded code into a shell, sending sensitive data to
  external endpoints, and destroying session files. It waves through local edits,
  lock-file installs, read-only requests and pushes to your own branch.
- **★ The classifier checks intent, not correctness.** Ask for a refactor of
  authentication and get broken authentication, and it waves it through, because
  broken is not dangerous.
- **So pair auto mode with a Stop hook that runs your tests.** One guards intent
  before each action, the other guards correctness after the turn.
- **Don't ask** is for unattended runs (CI, scheduled jobs, overnight batches). Only
  pre-approved tools are allowed; everything else is auto-denied with no prompt, so
  the pipeline keeps moving instead of hanging on an approval no one is there to give.
- **Bypass permissions** only inside an isolated container or VM.

# Lecture 4 - Permission Modes

### The six permission modes

| Mode | What runs without asking | Reach for it when |
|---|---|---|
| **Manual** | Reads only. Everything else asks first. | You want maximum control |
| **Accept edits** | Reads, file edits, common file system bash commands | Iterating on code you review after the fact |
| **Plan** | Reads only. Researches and proposes, edits nothing. | Scoping work before execution |
| **Auto** | Everything, with a classifier reviewing each action first | Hands-off runs you are still nominally around for |
| **Don't ask** | Only pre-approved tools. Everything else auto-denied, no prompt. | CI, scheduled jobs, overnight batches |
| **Bypass permissions** | Everything, no checks at all | **Only** inside an isolated container or VM |

**Cycling:** press **shift-tab** to cycle the everyday modes (manual, accept edits,
plan, auto). The status bar at the bottom always shows the current mode.

Bypass permissions is the equivalent of the `dangerously-skip-permissions` flag.

### ★ How auto mode works, and what it cannot do

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1240 680" width="880" height="483" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img"><title>The two guards on an unattended run</title><desc>Auto mode's classifier guards intent before each action; a Stop hook guards correctness after the turn.</desc><rect width="1240" height="680" fill="#FFFFFF"/><defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs><rect x="420" y="60" width="400" height="110" rx="8" fill="#EBD6F8" stroke="#A81FEE" stroke-width="3"/><text x="620" y="122" font-size="19" fill="#000000" text-anchor="middle"><tspan x="620" dy="0" font-weight="700">Unattended run in auto mode</tspan></text><path d="M 620 170 L 620 210" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round"/><line x1="300" y1="210" x2="940" y2="210" stroke="#000000" stroke-width="3"/><line x1="300" y1="210" x2="300" y2="262" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><line x1="940" y1="210" x2="940" y2="262" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="209.48" y="229" width="181.04" height="24" fill="#FFFFFF"/><text x="300" y="248" font-size="16" fill="#5F6368" text-anchor="middle">before each action</text><rect x="844.84" y="229" width="190.32" height="24" fill="#FFFFFF"/><text x="940" y="248" font-size="16" fill="#5F6368" text-anchor="middle">after the turn ends</text><rect x="60" y="270" width="480" height="170" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/><text x="300" y="320" font-size="19" fill="#000000" text-anchor="middle"><tspan x="300" dy="0" font-weight="700">Classifier reviews intent</tspan><tspan x="300" dy="28">Blocks prod deploys, force push,</tspan><tspan x="300" dy="28">piping downloads into a shell,</tspan><tspan x="300" dy="28">sending data to outside endpoints</tspan></text><rect x="700" y="270" width="480" height="170" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/><text x="940" y="320" font-size="19" fill="#000000" text-anchor="middle"><tspan x="940" dy="0" font-weight="700">Stop hook checks correctness</tspan><tspan x="940" dy="28">Runs your tests and refuses</tspan><tspan x="940" dy="28">to end the turn on a failure.</tspan><tspan x="940" dy="28">Exit 2 feeds the failure back.</tspan></text><line x1="300" y1="440" x2="300" y2="482" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><line x1="940" y1="440" x2="940" y2="482" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="60" y="490" width="480" height="130" rx="8" fill="#E3F6E3" stroke="#5CBF63" stroke-width="3"/><text x="300" y="534" font-size="19" fill="#000000" text-anchor="middle"><tspan x="300" dy="0" font-weight="700">It never judges correctness.</tspan><tspan x="300" dy="28">Broken auth is waved through,</tspan><tspan x="300" dy="28">because broken is not dangerous.</tspan></text><rect x="700" y="490" width="480" height="130" rx="8" fill="#ECF3FC" stroke="#6BB8F5" stroke-width="3"/><text x="940" y="534" font-size="19" fill="#000000" text-anchor="middle"><tspan x="940" dy="0" font-weight="700">It never judges danger.</tspan><tspan x="940" dy="28">It only tells you whether</tspan><tspan x="940" dy="28">the code actually runs.</tspan></text></svg>

*The classifier guards intent before each action; the Stop hook guards correctness after the turn.*

In auto mode Claude runs on its own, but a **separate classifier model reviews each
action before it executes**. The classifier guards **intent**, watching for moves that
escalate beyond what you actually asked for.

| It blocks | It waves through |
|---|---|
| Production deploys and migrations | Local edits inside your project |
| Force pushing, or piping downloaded code straight into a shell | Installing dependencies from your lock file |
| Sending sensitive data to external endpoints | Read-only requests |
| Destroying files that exist for the session | Pushing to your own branch |

**★ The classifier checks intent, not correctness.** Ask Claude to refactor
authentication and it writes broken authentication, and the classifier waves it
through, because broken is not dangerous.

That is why you pair auto mode with a **Stop hook that runs your tests**. One guards
intent before each action, the other guards correctness after Claude finishes.

Auto mode's guardrails are still evolving, so check the docs for the current block and
allow lists.

### Don't ask, for unattended runs

Reach for **don't ask** whenever no human is around to approve prompts: CI pipelines,
scheduled jobs, overnight batches. Only pre-approved tools are allowed and anything
off that list is auto-denied with no prompt.

That is the whole point. Your pipeline keeps moving instead of hanging on an approval
no one is there to give.

## Executive Summary

A CLAUDE.md rule is a request. **A hook is deterministic code that runs at a fixed
point in the loop, so it guarantees behaviour instead of hoping for it.**

- Around **30 hook events** fire per session. The ones worth knowing: **PreToolUse**
  (before a tool call, the enforcement primitive), **PostToolUse** (after a successful
  call, where auto-format and auto-lint go), **Stop** (Claude wants to end its turn;
  you can refuse), **PreCompact / PostCompact**, **InstructionsLoaded**, **SessionStart**.
- **★ Gotcha:** to re-inject context after compaction, use **SessionStart with the
  compact matcher**, not PostCompact. That is the one whose output gets back into the
  conversation.
- **PreToolUse talks back in JSON, exit 0.** `permissionDecision` takes `allow`,
  `deny` or `ask`. A rarely used fourth value, `defer`, applies only to non-interactive
  `-p` runs.
- **`updatedInput` rewrites the call instead of blocking it.** This is how you redact a
  secret out of a bash command and still let it run. It **replaces the whole input
  object**, so echo back the fields you are not changing.
- **★ Exit codes: 2 blocks, 1 does not.** Exit 1 feels like an error but Claude runs
  the command anyway. If you meant to stop something, exit 2.

# Lecture 5 - Hooks

### The hook events worth knowing

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1440 850" width="880" height="519" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img"><title>Where the main hook events sit in the loop</title><desc>A cycle from SessionStart through UserPromptSubmit, PreToolUse, PostToolUse and Stop.</desc><rect width="1440" height="850" fill="#FFFFFF"/><defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs><rect x="60" y="60" width="340" height="170" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/><text x="230" y="110" font-size="19" fill="#000000" text-anchor="middle"><tspan x="230" dy="0" font-weight="700">SessionStart</tspan><tspan x="230" dy="28">Primes the environment.</tspan><tspan x="230" dy="28">Use the compact matcher</tspan><tspan x="230" dy="28">to re-inject state.</tspan></text><rect x="940" y="60" width="340" height="170" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/><text x="1110" y="124" font-size="19" fill="#000000" text-anchor="middle"><tspan x="1110" dy="0" font-weight="700">UserPromptSubmit</tspan><tspan x="1110" dy="28">Plain stdout on exit 0</tspan><tspan x="1110" dy="28">is added to context.</tspan></text><rect x="940" y="340" width="340" height="170" rx="8" fill="#CFE3FA" stroke="#0B6BE8" stroke-width="3"/><text x="1110" y="390" font-size="19" fill="#000000" text-anchor="middle"><tspan x="1110" dy="0" font-weight="700">PreToolUse</tspan><tspan x="1110" dy="28">The enforcement primitive.</tspan><tspan x="1110" dy="28">allow, deny, ask, or</tspan><tspan x="1110" dy="28">rewrite the call.</tspan></text><rect x="940" y="620" width="340" height="170" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/><text x="1110" y="684" font-size="19" fill="#000000" text-anchor="middle"><tspan x="1110" dy="0" font-weight="700">PostToolUse</tspan><tspan x="1110" dy="28">Auto-format, auto-lint.</tspan><tspan x="1110" dy="28">Too late to block.</tspan></text><rect x="60" y="620" width="340" height="170" rx="8" fill="#DDF7F1" stroke="#12807A" stroke-width="3"/><text x="230" y="670" font-size="19" fill="#000000" text-anchor="middle"><tspan x="230" dy="0" font-weight="700">Stop</tspan><tspan x="230" dy="28">Refuse the end of turn.</tspan><tspan x="230" dy="28">Exit 2 sends it</tspan><tspan x="230" dy="28">back to Claude.</tspan></text><line x1="400" y1="145" x2="932" y2="145" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><line x1="1110" y1="230" x2="1110" y2="332" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><line x1="1110" y1="510" x2="1110" y2="612" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><line x1="940" y1="705" x2="408" y2="705" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><line x1="230" y1="620" x2="230" y2="238" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><path d="M 1280 705 L 1370 705 L 1370 425 L 1288 425" fill="none" stroke="#9AA0A6" stroke-width="3" stroke-linejoin="round" stroke-dasharray="10 8" marker-end="url(#ah)"/><rect x="1298.04" y="551" width="143.92" height="24" fill="#FFFFFF"/><text x="1370" y="570" font-size="16" fill="#5F6368" text-anchor="middle">next tool call</text><rect x="490" y="345" width="400" height="160" rx="8" fill="#EEEEFC" stroke="#8B8BF5" stroke-width="3"/><text x="690" y="404" font-size="19" fill="#000000" text-anchor="middle"><tspan x="690" dy="0" font-weight="700">Around 30 hook events</tspan><tspan x="690" dy="28">fire in a session.</tspan><tspan x="690" dy="28">These five cover most work.</tspan></text></svg>

*The five events that cover most work, positioned at the points in the loop where you would want to step in.*

| Event | Fires | Typical use |
|---|---|---|
| **PreToolUse** | Before a tool call | **The enforcement primitive.** The only one that can stop something before it happens. |
| **PostToolUse** | After a successful tool call | Auto-formatting, auto-lint |
| **Stop** | Claude wants to end its turn | Refuse and say "you are not done yet". `SubagentStop` matches for sub-agents. |
| **PreCompact / PostCompact** | Around compaction | Bookkeeping either side of a compact |
| **InstructionsLoaded** | A CLAUDE.md or rule file loads | Auditing what actually made it into context |
| **SessionStart** | Session start | Priming the environment. Use the `startup` source for fresh starts only. |

**★ The gotcha that trips people up:** to re-inject context **after** compaction, do
not use `PostCompact`. Use **`SessionStart` with the `compact` matcher**. That is the
one whose output actually gets back into the conversation.

### ★ PreToolUse: returning a decision as JSON

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1380 830" width="880" height="529" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img"><title>What a PreToolUse hook can return</title><desc>A PreToolUse hook returns allow, deny or ask, or rewrites the call with updatedInput.</desc><rect width="1380" height="830" fill="#FFFFFF"/><defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs><rect x="60" y="370" width="300" height="140" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/><text x="210" y="433" font-size="19" fill="#000000" text-anchor="middle"><tspan x="210" dy="0" font-weight="700">Claude requests</tspan><tspan x="210" dy="28">a tool call</tspan></text><line x1="360" y1="440" x2="462" y2="440" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/><rect x="470" y="370" width="340" height="140" rx="8" fill="#C6F0C6" stroke="#018001" stroke-width="3"/><text x="640" y="433" font-size="19" fill="#000000" text-anchor="middle"><tspan x="640" dy="0" font-weight="700">PreToolUse hook</tspan><tspan x="640" dy="28">prints JSON, exits 0</tspan></text><rect x="900" y="60" width="420" height="550" rx="8" fill="none" stroke="#00C9A7" stroke-width="2" stroke-dasharray="8 6"/><rect x="912" y="70" width="179.4" height="21" fill="#FFFFFF"/><text x="918" y="86" font-size="16" fill="#5F6368">permissionDecision</text><line x1="810" y1="440" x2="865" y2="440" stroke="#000000" stroke-width="3"/><path d="M 865 440 L 865 175 L 912 175" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round" marker-end="url(#ah)"/><rect x="920" y="110" width="380" height="130" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/><text x="1110" y="168" font-size="19" fill="#000000" text-anchor="middle"><tspan x="1110" dy="0" font-weight="700">allow</tspan><tspan x="1110" dy="28">the call runs</tspan></text><path d="M 865 440 L 865 345 L 912 345" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round" marker-end="url(#ah)"/><rect x="920" y="280" width="380" height="130" rx="8" fill="#FCEEEB" stroke="#DC0A0A" stroke-width="3"/><text x="1110" y="338" font-size="19" fill="#000000" text-anchor="middle"><tspan x="1110" dy="0" font-weight="700">deny</tspan><tspan x="1110" dy="28">the call is stopped</tspan></text><path d="M 865 440 L 865 515 L 912 515" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round" marker-end="url(#ah)"/><rect x="920" y="450" width="380" height="130" rx="8" fill="#FCF2E3" stroke="#C4520A" stroke-width="3"/><text x="1110" y="508" font-size="19" fill="#000000" text-anchor="middle"><tspan x="1110" dy="0" font-weight="700">ask</tspan><tspan x="1110" dy="28">hand it to the user</tspan></text><path d="M 865 440 L 865 705 L 912 705" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round" marker-end="url(#ah)"/><rect x="920" y="640" width="380" height="130" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/><text x="1110" y="684" font-size="19" fill="#000000" text-anchor="middle"><tspan x="1110" dy="0" font-weight="700">updatedInput</tspan><tspan x="1110" dy="28">rewrite the call: redact</tspan><tspan x="1110" dy="28">the secret, still run it</tspan></text><rect x="959.16" y="781" width="301.68" height="24" fill="#FFFFFF"/><text x="1110" y="800" font-size="16" fill="#5F6368" text-anchor="middle">a sibling field, not a decision</text></svg>

*permissionDecision picks one of three outcomes; updatedInput is a separate field that rewrites the call.*

PreToolUse talks back to Claude by **printing JSON and exiting 0**. The key field is
`permissionDecision`:

| Value | Effect |
|---|---|
| `allow` | Let the call through |
| `deny` | Stop the call |
| `ask` | Hand it back to the user to decide |
| `defer` | Rare. Only for non-interactive `-p` runs where a calling process pauses the tool and resumes it later. |

In [ ]:
{
  "hookSpecificOutput": {
    "hookEventName": "PreToolUse",
    "permissionDecision": "deny",
    "permissionDecisionReason": "...",
    "updatedInput": {
      "command": "..."
    }
  }
}

**★ Note `updatedInput`.** Instead of blocking a call, you can rewrite it. That is how
you redact a secret out of a bash command and still let it run.

**One catch:** `updatedInput` replaces the **whole** input object, so you have to echo
back the fields you are not changing or you will lose them.

### ★ Exit codes, for hooks that do not return JSON

| Exit code | Meaning | Detail |
|---|---|---|
| **0** | Success | If stdout is JSON, Claude parses it. Plain text is ignored on most events, but on **SessionStart**, **UserPromptSubmit** and **UserPromptExpansion** plain text is added to context. That is what makes a state-preserver hook work. |
| **2** | **Blocking error** | stderr is fed back to Claude as context. This is the blocking exit code almost everywhere. |
| **1 (and anything else)** | **Non-blocking** | stderr is logged and Claude carries on. |

**★ The one that catches people out is exit code 1.** It feels like an error, but it
does not block. Claude runs the command anyway. **If you meant to stop something,
exit 2, not 1.**

Two more wrinkles:

- Exit 2 can even block **Stop**, which is how you tell Claude it is not done.
- **PostToolUse** fires after the tool already ran, so blocking there is too late to
  stop the call, though it can still feed text back to Claude.
- A few events ignore blocking entirely, such as **Notification** and **SessionStart**.
  They show your stderr and carry on regardless.

### ★ A real guardrail: redact instead of block

Say you want a PreToolUse guardrail on the Bash tool. The **matcher** picks the tool to
watch, and an optional **if** clause narrows it to a specific command.

The obvious move is `deny`, stopping a dangerous call. The lesser-known and more
interesting move is `updatedInput`, which rewrites it.

In practice: Claude is asked to run a command containing a live-looking secret. The
hook intercepts it, spots the `sk_live_` pattern, and swaps it for a placeholder before
the command executes.

**The command still ran. The work still got done. The secret never made it through.**
That is the difference between blocking and redacting, and a hook enforces it every
single time.

### Preserving state across a compact

When Claude compacts a long conversation it drops a lot of detail. A **SessionStart
hook with the `compact` matcher** runs right after compaction. Have it print a short
summary of the files you have been working on. That summary goes back into context, so
Claude picks up where it left off instead of starting cold.

## Executive Summary

A spectrum from "build nothing" to "full control", for work you no longer want to kick
off by hand.

1. **Routines.** A saved prompt bundled with a repo and connectors, running on
   **Anthropic's** infrastructure. Triggers: cron, HTTP POST to its API endpoint, or a
   GitHub event. Create from `claude.ai/code/routines` or `/schedule` in the terminal.
   **★ Three limits:** research preview; a recurring schedule runs **at most hourly**;
   each run starts from a **fresh clone of your default branch** and can only push to
   **`claude/` prefixed branches** unless you loosen it per repo.
2. **Headless mode (`-p` / `--print`).** One-shot, no interactive UI, reads stdin and
   writes stdout. **★ It skips auto-discovery of hooks, skills, plugins, MCP servers
   and CLAUDE.md**, which is why startup is much faster.
3. **`--bare`.** Deterministic mode, for when CI needs the same result every run.
4. **Agent SDK.** Claude Code as a library inside your TypeScript or Python app.

**Start with routines. Drop down the spectrum only when the job actually needs the
extra control.**

# Lecture 6 - Routines and Headless

### The spectrum

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1240 700" width="880" height="497" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img"><title>The automation spectrum</title><desc>Four options from managed routines through headless mode and bare mode to the Agent SDK.</desc><rect width="1240" height="700" fill="#FFFFFF"/><defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs><text x="95" y="124" font-size="46" font-weight="700" fill="#000000">1</text><rect x="60" y="140" width="460" height="170" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/><text x="290" y="190" font-size="19" fill="#000000" text-anchor="middle"><tspan x="290" dy="0" font-weight="700">Routines</tspan><tspan x="290" dy="28">A prompt, a repo and connectors,</tspan><tspan x="290" dy="28">run on Anthropic's infrastructure.</tspan><tspan x="290" dy="28">Cron, HTTP POST or GitHub event.</tspan></text><text x="695" y="124" font-size="46" font-weight="700" fill="#000000">2</text><rect x="660" y="140" width="460" height="170" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/><text x="890" y="190" font-size="19" fill="#000000" text-anchor="middle"><tspan x="890" dy="0" font-weight="700">Headless mode, the -p flag</tspan><tspan x="890" dy="28">One shot, pipes like a shell tool.</tspan><tspan x="890" dy="28">Skips auto-discovery of hooks, skills,</tspan><tspan x="890" dy="28">plugins, MCP servers and CLAUDE.md.</tspan></text><line x1="520" y1="225" x2="652" y2="225" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/><path d="M 890 310 L 890 375 L 290 375 L 290 452" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round" marker-end="url(#ah)"/><text x="95" y="444" font-size="46" font-weight="700" fill="#000000">3</text><rect x="60" y="460" width="460" height="170" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/><text x="290" y="510" font-size="19" fill="#000000" text-anchor="middle"><tspan x="290" dy="0" font-weight="700">The --bare flag</tspan><tspan x="290" dy="28">Deterministic mode: the same</tspan><tspan x="290" dy="28">result on every single run.</tspan><tspan x="290" dy="28">The right choice inside CI.</tspan></text><text x="695" y="444" font-size="46" font-weight="700" fill="#000000">4</text><rect x="660" y="460" width="460" height="170" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/><text x="890" y="510" font-size="19" fill="#000000" text-anchor="middle"><tspan x="890" dy="0" font-weight="700">The Agent SDK</tspan><tspan x="890" dy="28">Claude Code as a library inside</tspan><tspan x="890" dy="28">your own TypeScript or Python app.</tspan><tspan x="890" dy="28">Same engine, called from your product.</tspan></text><line x1="520" y1="545" x2="652" y2="545" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="194.84" y="653" width="190.32" height="24" fill="#FFFFFF"/><text x="290" y="672" font-size="16" fill="#5F6368" text-anchor="middle">you build the least</text><rect x="790.2" y="653" width="199.6" height="24" fill="#FFFFFF"/><text x="890" y="672" font-size="16" fill="#5F6368" text-anchor="middle">you control the most</text></svg>

*Four ways to hand off repeat work, from managed infrastructure to a library inside your own product.*

### Routines: a saved prompt that runs in the cloud

A routine bundles three things (**a prompt, the repository it works on, and any
connectors it needs**) and runs that bundle in the cloud whenever it is triggered.

The key part: **the infrastructure is Anthropic's.** No machine of yours staying on
overnight, no workflow file to maintain.

**Triggers:**

- A **cron schedule**, like every morning at 9am.
- An **HTTP POST** to its API endpoint, so your own code can kick it off.
- A **GitHub event**, like a new pull request landing.

**Good fits:** a morning dependency audit, a PR triager that fires on new pull
requests, a daily scan of Sentry tickets to work out what is most urgent.

**Two ways to create one:**

1. From the web at `claude.ai/code/routines`. Name it, write the instructions, pick a
   repository, choose a trigger.
2. From inside Claude Code with `/schedule`, described in plain language.

In [ ]:
/schedule daily dependency audit at 9am

### ★ Three limits before you rely on routines

1. **Routines are a research preview.** Behaviour and limits will keep moving.
2. **A recurring schedule runs at most hourly.** If you need something more frequent,
   routines are not the tool.
3. **Each run starts from a fresh clone of your default branch and can only push to
   `claude/` prefixed branches** unless you loosen that per repo. This is the guardrail
   that keeps an autonomous run from rewriting main.

### Headless mode: the -p flag

`-p` (short for `--print`) runs Claude Code as a one-shot command with no interactive
UI. It reads stdin and writes stdout, so it pipes like any other shell tool.

**★ Worth knowing:** `-p` **skips auto-discovery** of hooks, skills, plugins, MCP
servers and the CLAUDE.md file. You get Claude plus the tools you allow explicitly, and
nothing the local environment happens to load. The upside is much faster startup.

In [ ]:
claude -p "summarize the changes in this diff"

### Getting structured output back

Pair a **JSON schema** with the JSON output format and Claude constrains its output to
match. The object lands in the **`structured_output`** field of the JSON response, so
you can pull it out with `jq` and pipe it into a database or another script.

In [ ]:
claude -p "Extract the exported function names from src/core/style.js" \
  --output-format json \
  --json-schema '{"type":"object","properties":{"functions":{"type":"array","items":{"type":"string"}}},"required":["functions"]}' \
  | jq '.structured_output.functions'

### Multi-step automation with sessions

For work spanning multiple steps, do not cram everything into one command. Capture the
session ID from the JSON output and resume it later. One script kicks off the work,
another resumes it with full context: handy when the first pass produces a plan and the
second carries it out.

In [ ]:
claude --resume "$(jq -r .session_id /tmp/plan.json)"

### Deterministic runs for CI, and the Agent SDK

**`--bare`** gives you deterministic mode: repeatable, predictable output rather than
anything that varies run to run. The right choice inside a pipeline.

**The Agent SDK** embeds Claude Code inside your own TypeScript or Python application.
Both languages expose a `query()` function and the same primitives as the CLI. You pass
a prompt plus options (`allowedTools` to control what Claude can do, a system prompt,
and a permission mode), then iterate over the messages Claude streams back. Same engine
as the CLI, callable from inside your product.

**Decision guide:**

| Reach for | When |
|---|---|
| **Routines** | The default for repeat work. Nothing for you to host. |
| **Headless `-p`** | The job needs your pipeline and you want to pipe data through a script |
| **`--bare`** | CI needs the same results every single run |
| **Agent SDK** | The work belongs inside your own product |

## Executive Summary

Two ways to put Claude on a pull request, solving different problems.

- **Code Review (managed).** An Anthropic-hosted service reviewing PRs through the
  Claude GitHub app. An org admin enables it from Claude Code admin settings, installs
  the app, picks repos and chooses timing: **once when a PR opens, on every push, or
  only on `@claude review`**. Review agents analyse the diff **against the full
  codebase**, not the changed lines in isolation, then post inline comments tagged by
  severity with a summary table in the check run, deduplicated and ranked.
- **★ Its boundaries:** it **never approves or blocks** a PR, there is **no managed
  autofix**, and it is a research preview on team and enterprise plans. Apply findings
  locally with **`/code-review --fix`**.
- **GitHub Action (do it yourself).** For when the job goes beyond review: implementing
  changes from a comment, scheduled reports, any GitHub event. Set up with
  **`/install-github-app`** (needs repo admin). The action is
  **`anthropics/claude-code-action@v1`**.
- Tune the run with **`claude_args`**: `--max-turns` caps the agent loop, a permission
  mode that will not stop and ask, and a minimal allowed-tools list.

**Start with the managed service. Move to the action the moment you need Claude to
actually do something in CI, not just comment on it.**

# Lecture 7 - GitHub Actions and Code Review

### Managed vs do-it-yourself

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1300 800" width="880" height="542" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img"><title>Two ways to put Claude on a pull request</title><desc>The managed Code Review service on the left, the do-it-yourself GitHub Action on the right.</desc><rect width="1300" height="800" fill="#FFFFFF"/><defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs><rect x="60" y="60" width="560" height="110" rx="8" fill="#EBD6F8" stroke="#A81FEE" stroke-width="3"/><text x="340" y="122" font-size="19" fill="#000000" text-anchor="middle"><tspan x="340" dy="0" font-weight="700">Code Review, the managed path</tspan></text><line x1="340" y1="170" x2="340" y2="222" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="60" y="230" width="560" height="130" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/><text x="340" y="288" font-size="19" fill="#000000" text-anchor="middle"><tspan x="340" dy="0" font-weight="700">Enable it once from the</tspan><tspan x="340" dy="28">Claude Code admin settings</tspan></text><line x1="340" y1="360" x2="340" y2="402" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="60" y="410" width="560" height="130" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/><text x="340" y="468" font-size="19" fill="#000000" text-anchor="middle"><tspan x="340" dy="0" font-weight="700">Review agents read the diff</tspan><tspan x="340" dy="28">against the whole codebase</tspan></text><line x1="340" y1="540" x2="340" y2="582" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="60" y="590" width="560" height="150" rx="8" fill="#ECF3FC" stroke="#6BB8F5" stroke-width="3"/><text x="340" y="644" font-size="19" fill="#000000" text-anchor="middle"><tspan x="340" dy="0" font-weight="700">Never approves or blocks a PR.</tspan><tspan x="340" dy="28">No managed autofix: apply findings</tspan><tspan x="340" dy="28">locally with /code-review --fix</tspan></text><rect x="680" y="60" width="560" height="110" rx="8" fill="#F9D5E7" stroke="#D6009A" stroke-width="3"/><text x="960" y="122" font-size="19" fill="#000000" text-anchor="middle"><tspan x="960" dy="0" font-weight="700">The GitHub Action, do it yourself</tspan></text><line x1="960" y1="170" x2="960" y2="222" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="680" y="230" width="560" height="130" rx="8" fill="#DDF7F1" stroke="#12807A" stroke-width="3"/><text x="960" y="288" font-size="19" fill="#000000" text-anchor="middle"><tspan x="960" dy="0" font-weight="700">Run /install-github-app,</tspan><tspan x="960" dy="28">then add a workflow file</tspan></text><line x1="960" y1="360" x2="960" y2="402" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="680" y="410" width="560" height="130" rx="8" fill="#EEEEFC" stroke="#5A4FF0" stroke-width="3"/><text x="960" y="468" font-size="19" fill="#000000" text-anchor="middle"><tspan x="960" dy="0" font-weight="700">anthropics/claude-code-action@v1</tspan><tspan x="960" dy="28">on @claude, cron, any event</tspan></text><line x1="960" y1="540" x2="960" y2="582" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="680" y="590" width="560" height="150" rx="8" fill="#EEEEFC" stroke="#8B8BF5" stroke-width="3"/><text x="960" y="644" font-size="19" fill="#000000" text-anchor="middle"><tspan x="960" dy="0" font-weight="700">Claude actually does the work.</tspan><tspan x="960" dy="28">Tune it with claude_args:</tspan><tspan x="960" dy="28">--max-turns, mode, allowed tools</tspan></text></svg>

*Code Review is a service you turn on; the GitHub Action is CI you wire up when review alone is not enough.*

### The managed path: Code Review

An Anthropic-hosted service that reviews pull requests through the Claude GitHub app.
Nothing for you to build or host.

**Setup (org admin):** Claude Code admin settings has a **Code review** section with a
**Configure** button. From there the admin installs the Claude GitHub app, picks which
repos it watches, and decides when it runs.

**Timing options:**

1. Once when a PR opens
2. On every push to the PR
3. Only when someone comments `@claude review`

**What happens:** review agents analyse the diff **against your full codebase**, not
just the changed lines in isolation. Findings post as inline comments on the specific
lines, tagged by severity, with a summary table in the check run. It **deduplicates and
ranks** them, so you read a handful of real issues rather than a wall of nitpicks.

### ★ What Code Review will and will not do

| Boundary | Detail |
|---|---|
| **Never approves or blocks the PR** | The judgment call stays with a human. Claude flags, you decide. |
| **No managed autofix** | The service posts findings only. |
| **Research preview** | Available on team and enterprise plans. Expect behaviour to keep moving. |

Since there is no autofix in the service, applying a finding is a **local** move. From
your own terminal, `/code-review` reviews a diff and its `--fix` flag applies the
findings to your working tree.

**The flow: Claude finds it in the PR, you pull it down and fix it locally.**

### The do-it-yourself path: the GitHub Action

Code Review handles review. When the job goes **beyond** review (implementing changes
from a comment, running scheduled reports, anything you would normally write a workflow
for) you reach for the action. It runs on PR comments, scheduled jobs, and any GitHub
event.

**Setup:** run **`/install-github-app`** inside Claude Code. You need repo admin. The
slash command walks you through installing the GitHub app and setting the Anthropic API
key secret on the repo.

The action is **`anthropics/claude-code-action@v1`**.

| Input | Notes |
|---|---|
| `anthropic_api_key` | Optional |
| `github_token` | Defaults to `secrets.GITHUB_TOKEN` |
| `trigger_phrase` | What the action listens for in comments. Defaults to `@claude`. |
| `use_bedrock` / `use_vertex` | Switch providers if you are on Bedrock or Vertex |
| `prompt` | The instruction for the run |
| `claude_args` | CLI arguments passed straight through to Claude Code |

### A workflow that responds to @claude

Drop this into `.github/workflows/claude.yaml` and it listens for `@claude` on PR and
issue comments. Someone writes "@claude implement the spec in the linked Linear issue"
and the action picks it up: Claude pushes commits and posts comments describing what it
did. The same action also works on a cron trigger for a daily rollup, and you can add
`workflow_dispatch` to kick it off manually from the Actions tab.

In [ ]:
- uses: anthropics/claude-code-action@v1
  with:
    anthropic_api_key: ${{ secrets.ANTHROPIC_API_KEY }}
    github_token: ${{ secrets.GITHUB_TOKEN }}
    trigger_phrase: "@claude"
    prompt: "Your instructions here"
    claude_args: "--max-turns 5 --model claude-sonnet-5"

**★ Tuning with `claude_args`:**

- **`--max-turns 5`** puts a hard cap on the agent loop, so it cannot run forever.
- **Permission mode.** For an unattended job you want one that will not stop and ask,
  since there is no one there to answer.
- **Allowed tools.** Give the job exactly what it needs and nothing more. For a report,
  that means read-only.

## Executive Summary

**Verify in proportion to how much rope you gave the run.** Watched the messages scroll
by in a short session? A glance is enough. Unattended run, or a CI job with nobody in
the loop? Nobody saw what happened, so you have to reconstruct it after the fact.

1. **Keep unattended runs in auto mode, not bypass permissions.** The classifier still
   reviews each action for danger. But it **never judges whether the code is correct**,
   so your verification bar does not move.
2. **★ Start with the diff, not the summary.** The trap is a tidy summary that reads
   perfectly fine while the actual diff touched a file you did not expect. Run
   `/code-review` to walk the changes, then put your own eyes on `git diff`. Read the
   files that were part of the plan first, then look for anything outside it.
   **A clean write-up is not proof of clean code.**
3. **★ Turn tests into a gate, not a promise.** Do not trust that Claude ran them. A
   **Stop hook** runs your tests and refuses to end the turn on failure; a
   **PostToolUse hook** lints and type checks after every edit. **Exit 2** feeds the
   failure straight back so Claude fixes it unprompted, on every run.
4. **Get a cold second opinion.** A fresh session or sub-agent with no memory of how the
   code was built catches what the original run talked itself past.
5. **Verify headless runs by their JSON result and exit code.**

# Lecture 8 - Trust It: Verifying Unsupervised Runs

### ★ Verify in proportion to how little you watched

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1240 690" width="880" height="490" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img"><title>Verifying an unsupervised run</title><desc>Four checks in order: read the diff, run code review, gate on tests, then get a cold second opinion.</desc><rect width="1240" height="690" fill="#FFFFFF"/><defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs><text x="95" y="124" font-size="46" font-weight="700" fill="#000000">1</text><rect x="60" y="140" width="460" height="170" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/><text x="290" y="204" font-size="19" fill="#000000" text-anchor="middle"><tspan x="290" dy="0" font-weight="700">Read the diff, not the summary</tspan><tspan x="290" dy="28">A tidy write-up can hide a file</tspan><tspan x="290" dy="28">you never expected it to touch.</tspan></text><text x="695" y="124" font-size="46" font-weight="700" fill="#000000">2</text><rect x="660" y="140" width="460" height="170" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/><text x="890" y="204" font-size="19" fill="#000000" text-anchor="middle"><tspan x="890" dy="0" font-weight="700">Run /code-review</tspan><tspan x="890" dy="28">It walks the changes and flags</tspan><tspan x="890" dy="28">issues before anything ships.</tspan></text><line x1="520" y1="225" x2="652" y2="225" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><path d="M 890 310 L 890 375 L 290 375 L 290 452" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round" marker-end="url(#ah)"/><text x="95" y="444" font-size="46" font-weight="700" fill="#000000">3</text><rect x="60" y="460" width="460" height="170" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/><text x="290" y="510" font-size="19" fill="#000000" text-anchor="middle"><tspan x="290" dy="0" font-weight="700">Make tests a gate</tspan><tspan x="290" dy="28">A Stop hook that exits 2 refuses</tspan><tspan x="290" dy="28">the turn and hands the failure</tspan><tspan x="290" dy="28">back to Claude to fix.</tspan></text><text x="695" y="444" font-size="46" font-weight="700" fill="#000000">4</text><rect x="660" y="460" width="460" height="170" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/><text x="890" y="510" font-size="19" fill="#000000" text-anchor="middle"><tspan x="890" dy="0" font-weight="700">Get a cold second opinion</tspan><tspan x="890" dy="28">A fresh sub-agent with no memory</tspan><tspan x="890" dy="28">of how the code was built catches</tspan><tspan x="890" dy="28">what the run talked itself past.</tspan></text><line x1="520" y1="545" x2="652" y2="545" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/></svg>

*Four checks in order; the gate on tests is the one that fires whether or not you remember to ask.*

The rule: **the less you watched, the more you verify.**

**Keep unattended runs in auto mode**, not bypass permissions. The classifier still
reviews each action for danger, which is a safety net worth keeping. But be clear about
what that net does: it **only flags dangerous actions and never judges whether the code
is correct**. Your verification bar stays exactly where it was.

### ★ Start with the diff, not the summary

Do not start with Claude's summary of what it did.

1. Run **`/code-review`** to walk the changes and flag issues.
2. Then put your own eyes on **`git diff`**.

**The trap:** a tidy summary that reads perfectly fine, while the actual diff touched a
file you honestly did not expect it to touch. The summary will not tell you that. The
diff will.

Read the files that were part of the plan first, then look for anything outside it.
**A clean write-up is not proof of clean code.**

### ★ Turn tests into a gate, not a promise

The real gate on an unsupervised run is whether the tests passed **and whether Claude
actually ran them or only claimed it did**. Do not leave that to trust. Wire it as a
hook so Claude cannot skip it.

| Hook | Job |
|---|---|
| **Stop hook** | Runs your tests and **refuses to end the turn** on a failure |
| **PostToolUse hook** | Lints and type checks after every edit |

**★ The key detail is the exit code.** A hook that exits with **`exit 2`** feeds the
failure straight back to Claude, which reads it and fixes it without you asking. Best of
all, the check fires on **every** run, whether or not you remember to ask for it.

### Get a cold second opinion

The sub-agent code review you would run before a pull request works here too. Open a
**fresh session or sub-agent** and have it review the changed code **with no memory of
how the code was built**.

Because it has no stake in the approach, it catches the things the original run talked
itself past. **A second reviewer with fresh eyes finds what the author rationalised
away.**

Do all of this and "Claude did it while I wasn't looking" no longer takes faith.

## Executive Summary

A plugin is **one installable unit** that packages a setup and moves it between people:
skills, subagents, hooks and MCP server configs, plus LSP servers, background monitors,
themes and a slice of `settings.json`. One version, one install.

- **Install:** `/plugin install org-name@plugin-name`, then `/reload-plugins`. For a
  team, add a private marketplace once with `/plugin marketplace add your-org/claude-plugins`
  and every install after that resolves through it.
- **★ Read before you install.** A plugin **runs code on your machine with your
  privileges**. Install it for its skills and you also get its PreToolUse and Stop hooks
  whether you read them or not. Claude Code shows what it will install and estimates
  context cost. **Reviewed is not the same as trusted.**
- **Components run alongside yours, they do not overwrite.** Hooks **stack**: a plugin's
  PreToolUse and yours both fire. Skills, agents and commands are **namespaced** under
  the plugin name.
- **★ A plugin's `settings.json` is honoured for only two keys**: the agent and subagent
  status line keys. Setting `agent` **promotes one of the plugin's subagents to the main
  thread**, along with its system prompt, tool restrictions and model. Enabling a plugin
  can change how Claude Code behaves by default.
- **Packaging your own needs no restructuring.** Same `.claude` shape, discovered by
  convention. The manifest at `.claude-plugin/plugin.json` is optional; **`name` is the
  only required field**.

# Lecture 9 - Plugins

### What a plugin is, and how to install one

The problem plugins solve: you build a great `.claude` directory with skills, subagents
and hooks, and then everyone copies and pastes files between machines and hopes they
stay in sync.

A plugin is **one installable unit** bundling skills, subagents, hooks and MCP server
configs, plus the longer tail (LSP servers, background monitors, themes, a slice of
`settings.json`). One version, one install.

In [ ]:
# Install a single plugin by name, then apply it
/plugin install org-name@plugin-name
/reload-plugins

# Better for a team: add a private marketplace once,
# and every install after that resolves through it
/plugin marketplace add your-org/claude-plugins

A marketplace gives you **centralized discovery, version tracking and updates in one
place** instead of scattered across everyone's laptop. Browse what is available from the
**Discover** tab.

### ★ Read before you install

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1300 900" width="880" height="609" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img"><title>A plugin's components run alongside yours</title><desc>Your own skills and hooks and an installed plugin's both fire on every matching tool call.</desc><rect width="1300" height="900" fill="#FFFFFF"/><defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs><rect x="60" y="60" width="520" height="350" rx="8" fill="none" stroke="#00C9A7" stroke-width="2" stroke-dasharray="8 6"/><rect x="72" y="70" width="179.4" height="21" fill="#FFFFFF"/><text x="78" y="86" font-size="16" fill="#5F6368">Your configuration</text><rect x="100" y="110" width="440" height="110" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/><text x="320" y="172" font-size="19" fill="#000000" text-anchor="middle"><tspan x="320" dy="0" font-weight="700">Your skills and agents</tspan></text><rect x="100" y="260" width="440" height="110" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/><text x="320" y="322" font-size="19" fill="#000000" text-anchor="middle"><tspan x="320" dy="0" font-weight="700">Your PreToolUse hook</tspan></text><rect x="720" y="60" width="520" height="350" rx="8" fill="none" stroke="#00C9A7" stroke-width="2" stroke-dasharray="8 6"/><rect x="732" y="70" width="160.8" height="21" fill="#FFFFFF"/><text x="738" y="86" font-size="16" fill="#5F6368">Installed plugin</text><rect x="760" y="110" width="440" height="110" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/><text x="980" y="172" font-size="19" fill="#000000" text-anchor="middle"><tspan x="980" dy="0" font-weight="700">Plugin skills, namespaced</tspan></text><rect x="760" y="260" width="440" height="110" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/><text x="980" y="322" font-size="19" fill="#000000" text-anchor="middle"><tspan x="980" dy="0" font-weight="700">The plugin's PreToolUse hook</tspan></text><path d="M 320 410 L 320 465" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round"/><path d="M 980 410 L 980 465" fill="none" stroke="#000000" stroke-width="3" stroke-linejoin="round"/><line x1="320" y1="465" x2="980" y2="465" stroke="#000000" stroke-width="3"/><line x1="650" y1="465" x2="650" y2="512" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/><rect x="390" y="520" width="520" height="140" rx="8" fill="#DADAFA" stroke="#5A4FF0" stroke-width="3"/><text x="650" y="569" font-size="19" fill="#000000" text-anchor="middle"><tspan x="650" dy="0" font-weight="700">Every matching tool call</tspan><tspan x="650" dy="28">Both hooks fire. Neither one</tspan><tspan x="650" dy="28">replaces or overrides the other.</tspan></text><line x1="650" y1="660" x2="650" y2="702" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/><rect x="340" y="710" width="620" height="130" rx="8" fill="#FCF2E3" stroke="#C4520A" stroke-width="3"/><text x="650" y="754" font-size="19" fill="#000000" text-anchor="middle"><tspan x="650" dy="0" font-weight="700">Read the details before you install.</tspan><tspan x="650" dy="28">A plugin runs code with your privileges,</tspan><tspan x="650" dy="28">and reviewed is not the same as trusted.</tspan></text></svg>

*A plugin's hooks do not replace yours; both fire on every matching tool call.*

**This is the part that matters most.** A plugin **runs code on your machine, with your
privileges**. Its hooks fire on every matching tool call. So if you install a plugin for
its skills, you **also get its PreToolUse and Stop hooks** whether you read them or not.

A community plugin could ship a Stop hook that calls out to a network endpoint every
time, and nothing in your configuration would warn you.

That is not a reason to avoid plugins. It is a reason to look first. Before installing,
check the plugin's details: Claude Code shows **what it will install**, estimates the
**context cost**, and states plainly that Anthropic does not control what is inside
third-party plugins.

**Where plugins come from:**

- The in-app submission form posts to the **community marketplace** after Anthropic's
  automated review.
- The **official marketplace** is curated on its own separate track.

**★ But reviewed is not the same as trusted.** Automated review catches some things, not
everything. Install plugins and add marketplaces only from sources you truly trust.

### ★ Components run alongside yours

A plugin does **not** overwrite your configuration. Its components run alongside your
own. Mostly good, with consequences:

| Component | Behaviour |
|---|---|
| **Hooks** | **They stack.** A plugin's PreToolUse and your own PreToolUse both fire on every tool call. Neither replaces the other. |
| **Skills, agents, commands** | **Namespaced** under the plugin name, so they never clash with yours |
| **`settings.json`** | Only a narrow one. Claude Code honours **just two keys**: the agent and subagent status line keys. |

**★ That agent key is worth a pause.** Setting it **promotes one of the plugin's
subagents to the main thread**, along with its system prompt, tool restrictions and
model. In other words, enabling the plugin can change how Claude Code behaves by
default. That is one of the main reasons to look before you even turn it on.

Once installed, you can see everything a plugin added, manage it, and uninstall it from
the plugin panel.

### Packaging your own plugin

**You do not have to restructure anything.** A plugin uses the same `.claude` shape you
already use, and Claude Code discovers components by convention:

- One folder per skill.
- One markdown file per subagent under `agents`.
- `hooks/hooks.json` and `.mcp.json` at the plugin root.

On top of that there is an **optional** manifest at `.claude-plugin/plugin.json`:

In [ ]:
{
  "name": "svg-splitter-review",
  "version": "0.1.0",
  "description": "Reviews the SVG Splitter repo",
  "author": {
    "name": "Lewis Menelaws"
  }
}

Leave the manifest out and Claude Code still discovers your components. But two details
matter:

- **`name` is the only required field.** It namespaces your skills as
  `company-name:skill-name`, keeping them from colliding with anyone else's.
- **Version it like any other dependency.** That is what makes updates and version
  tracking work across your team.

**The two rules that cover most of this:**

1. **When you use plugins, read before you install.** A plugin runs code with your
   privileges, so look at its hooks, agents and MCP servers first.
2. **When you build one, package your `.claude` the moment it works.** One manifest, one
   install, and the setup you trust reaches your entire team.